In [ ]:
import os
import re
import json
import time
import asyncio
import pathlib
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional
from abc import ABC, abstractmethod

from huggingface_hub import InferenceClient
from PIL import Image
from prompts import IMAGE_PROMPTS, TEXT_PROMPTS
from models import * 
from dotenv import load_dotenv

load_dotenv()

class CubeMTGenerator(Enum):
    IMAGE = "image"
    TEXT = "text"

@dataclass
class BaseConfig:
    prompts: list[str]
    output_root: pathlib.Path
    seeds: list[int] = field(default_factory=lambda: list(range(80)))
    batch_size: int = 8
    
    def __post_init__(self):
        self.output_root = pathlib.Path(self.output_root)

@dataclass
class ImageConfig(BaseConfig):
    model_id: str = "Qwen/Qwen-Image"
    provider: str = "fal-ai"
    width: int = 1024
    height: int = 1024
    guidance: float = 5.0
    steps: int = 28
    negative_prompt: str = "blurry, low quality, low resolution, artifacts"
    max_concurrency: int = 4


@dataclass
class TextConfig(BaseConfig):
    model_id: str = "google/gemma-2-2b-it"
    max_tokens: int = 512
    temperature: float = 0.8

def slugify(s: str, max_len: int = 60) -> str:
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9]+", "-", s)
    return re.sub(r"-+", "-", s).strip("-")[:max_len]

def get_client(config: BaseConfig) -> InferenceClient:
    
    api_key = os.environ.get("HF_TOKEN")
    if not api_key:
        raise RuntimeError("Missing HF_TOKEN environment variable")
    
    if isinstance(config, ImageConfig):
        return InferenceClient(provider=config.provider, api_key=api_key)
    return InferenceClient(api_key=api_key)


class Generator(ABC):
    def __init__(self, config: BaseConfig):
        self.config = config
        self.client = get_client(config)
    
    @abstractmethod
    def generate_one(self, prompt: str, seed: int, outdir: pathlib.Path) -> pathlib.Path:
        pass
    
    def get_output_path(self, prompt: str, seed: int) -> tuple[pathlib.Path, pathlib.Path]:
        pslug = slugify(prompt)
        prompt_dir = self.config.output_root / pslug
        round_idx = seed // self.config.batch_size
        outdir = prompt_dir / f"round-{round_idx:02d}"
        outdir.mkdir(parents=True, exist_ok=True)
        return prompt_dir, outdir


class TextGenerator(Generator):
    def __init__(self, config: TextConfig):
        super().__init__(config)
        self.config: TextConfig = config
    
    def generate_one(self, prompt: str, seed: int, outdir: pathlib.Path) -> pathlib.Path:
        completion = self.client.chat.completions.create(
            model=self.config.model_id,
            messages=[{"role": "user", "content": prompt}],
            seed=seed,
            temperature=self.config.temperature,
            max_tokens=self.config.max_tokens
        )
        
        content = completion.choices[0].message.content.strip()
        outpath = outdir / f"seed-{seed:02d}.json"
        
        with open(outpath, 'w', encoding='utf-8') as f:
            json.dump({
                "seed": seed,
                "prompt": prompt,
                "response": content,
                "model": self.config.model_id
            }, f, indent=2, ensure_ascii=False)
        
        return outpath
    
    def run(self):
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        print(f"Text Generation: {self.config.model_id}")
        
        for prompt_idx, prompt in enumerate(self.config.prompts, 1):
            print(f"\nPrompt {prompt_idx}/{len(self.config.prompts)}: {prompt[:60]}...")
            successful, failed = 0, 0
            
            for seed in self.config.seeds:
                _, outdir = self.get_output_path(prompt, seed)
                try:
                    self.generate_one(prompt, seed, outdir)
                    successful += 1
                    time.sleep(0.1)
                except Exception as e:
                    print(f"  [seed {seed:02d}] FAILED: {str(e)[:80]}")
                    failed += 1
                    time.sleep(1)
            
            print(f"  Summary: {successful}/{len(self.config.seeds)} successful")


class ImageGenerator(Generator):
    def __init__(self, config: ImageConfig):
        super().__init__(config)
        self.config: ImageConfig = config
    
    async def generate_one_async(
        self, prompt: str, seed: int, outdir: pathlib.Path, retries: int = 2
    ) -> pathlib.Path:
        last_exc: Optional[Exception] = None
        
        for attempt in range(retries + 1):
            try:
                img: Image.Image = await asyncio.to_thread(
                    self.client.text_to_image,
                    prompt,
                    model=self.config.model_id,
                    negative_prompt=self.config.negative_prompt,
                    height=self.config.height,
                    width=self.config.width,
                    num_inference_steps=self.config.steps,
                    guidance_scale=self.config.guidance,
                    seed=seed,
                )
                outpath = outdir / f"seed-{seed:02d}.png"
                img.save(outpath)
                return outpath
            except Exception as e:
                last_exc = e
                if attempt < retries:
                    await asyncio.sleep(1.5 ** attempt)
        raise last_exc
    
    def generate_one(self, prompt: str, seed: int, outdir: pathlib.Path) -> pathlib.Path:
        return asyncio.run(self.generate_one_async(prompt, seed, outdir))
    
    async def _worker(
        self, sem: asyncio.Semaphore, prompt: str, seed: int, prompt_dir: pathlib.Path
    ):
        _, outdir = self.get_output_path(prompt, seed)
        pslug = slugify(prompt)
        
        async with sem:
            try:
                path = await self.generate_one_async(prompt, seed, outdir)
                print(f"[{pslug}] saved {path.relative_to(prompt_dir)}")
            except Exception as e:
                print(f"[{pslug}] seed {seed} failed: {e}")
    
    async def run_async(self):
        self.config.output_root.mkdir(parents=True, exist_ok=True)
        sem = asyncio.Semaphore(self.config.max_concurrency)
        print(f"Image Generation: {self.config.model_id}")
        
        for prompt in self.config.prompts:
            prompt_dir, _ = self.get_output_path(prompt, 0)
            pslug = slugify(prompt)
            
            tasks = [
                asyncio.create_task(self._worker(sem, prompt, seed, prompt_dir))
                for seed in self.config.seeds
            ]
            
            total_batches = (len(self.config.seeds) + self.config.batch_size - 1) // self.config.batch_size
            for i in range(0, len(tasks), self.config.batch_size):
                batch_no = i // self.config.batch_size + 1
                print(f"\n== {pslug}: batch {batch_no}/{total_batches} ==")
                await asyncio.gather(*tasks[i:i + self.config.batch_size])
        
        print("\Run competed.")
    
    def run(self):
        try:
            asyncio.run(self.run_async())
        except RuntimeError as e:
            if "cannot be called from a running event loop" in str(e):
                import nest_asyncio
                nest_asyncio.apply()
                asyncio.run(self.run_async())
            else:
                raise

def run_image_generation(
    prompts: list[str],
    output_root: str,
    model_id: str = "Qwen/Qwen-Image",
    provider: str = "fal-ai",
    **kwargs
):
    config = ImageConfig(
        prompts=prompts,
        output_root=pathlib.Path(output_root),
        model_id=model_id,
        provider=provider,
        **kwargs
    )
    ImageGenerator(config).run()


def run_text_generation(
    prompts: list[str],
    output_root: str,
    model_id: str = "google/gemma-2-2b-it",
    **kwargs
):
    config = TextConfig(
        prompts=prompts,
        output_root=pathlib.Path(output_root),
        model_id=model_id,
        **kwargs
    )
    TextGenerator(config).run()


if __name__ == "__main__":
    GEN_TYPE = CubeMTGenerator.IMAGE
    CULTURAL_CONCEPT = "landmarks"
    OUTPUT_ROOT = "<LOCAL_OUTPUT_PATH>"
    
    if GEN_TYPE == CubeMTGenerator.IMAGE:
        prompts = IMAGE_PROMPTS[CULTURAL_CONCEPT]
        default_image_model = QWEN_IMAGE
        run_image_generation(prompts, OUTPUT_ROOT, 
                             model_id=QWEN_IMAGE["model_id"],
                             provider=QWEN_IMAGE["provider"])
    else:
        prompts = TEXT_PROMPTS[CULTURAL_CONCEPT]
        default_text_model = GOOGLE_GEMMA_2_2B_IT
        run_text_generation(prompts, 
                            OUTPUT_ROOT, 
                            model_id=GOOGLE_GEMMA_2_2B_IT)